# Tutorial 3: Marginal Effects

In Tutorial 1 we computed a basic average marginal effect for age. This tutorial goes deeper. We will learn how to compute marginal effects for continuous and categorical variables, how to evaluate them at different points, and how to use them with interaction terms.

## What you will learn

- Average Marginal Effect (AME), Marginal Effect at the Mean (MEM), and Marginal Effect at Representative values (MER)
- Discrete contrasts for categorical variables
- Discrete change for dummy variables
- Subgroup marginal effects via `atexog`

## Setup

We continue with the same fitted model and `Margins` object from the previous tutorials:

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from smmargins import Margins

rng = np.random.default_rng(7)
N = 5_000
df = pd.DataFrame({
    "age":    rng.normal(45, 12, N).clip(18, 90),
    "income": rng.lognormal(10.5, 0.4, N),
    "educ":   rng.choice(["hs", "college", "grad"], N, p=[0.4, 0.4, 0.2]),
    "female": rng.integers(0, 2, N),
})
eta = (-4.0 + 0.05 * df["age"] + 0.00001 * df["income"]
       + 0.8 * (df["educ"] == "college") + 1.4 * (df["educ"] == "grad")
       + 0.3 * df["female"] - 0.0004 * df["age"] * df["female"])
df["voted"] = (rng.uniform(0, 1, N) < 1 / (1 + np.exp(-eta))).astype(int)

fit = smf.logit("voted ~ age + income + C(educ) + female + age:female", data=df).fit(disp=False)
M = Margins(fit)

## Continuous variables: AME, MEM, and MER

### Average Marginal Effect (AME)

The AME is the default. It computes the marginal effect at each observation using that observation's actual covariate values, then averages:

In [ ]:
M.dydx("age").summary()

The AME answers: "What is the average effect of a one-year age increase across our sample?"

### Marginal Effect at the Mean (MEM)

The MEM sets all covariates to their means (or reference levels) and computes the marginal effect at that single point:

In [ ]:
M.dydx("age", at="mean").summary()

The MEM answers: "What is the effect of age for an average individual?"

### Marginal Effect at Representative values (MER)

The MER lets you choose specific values. Here we compute the marginal effect of age at ages 25, 45, and 65:

In [ ]:
M.dydx("age", atexog={"age": [25, 45, 65]}).summary()

The MER answers: "How does the effect of age differ at ages 25, 45, and 65?" Notice that the effect is slightly smaller at the extremes because the logistic curve flattens near 0 and 1.

## Categorical variables: discrete contrasts

For categorical variables like `educ`, `dydx()` computes pairwise contrasts between levels. By default it compares each level to the reference level (the first level alphabetically, `"college"` in our case because `C(educ)` uses treatment coding):

In [ ]:
M.dydx("educ").summary()

Each row is the difference in predicted probability between two education levels, averaged across the sample. For example, individuals with a graduate degree have a predicted probability of voting that is about 14.2 percentage points higher than those with a high school diploma.

You can change the reference level:

In [ ]:
M.dydx("educ", reference="hs").summary()

## Binary variables: discrete change

For binary (dummy) variables like `female`, `dydx()` computes the discrete change: the difference in predicted probability when moving from 0 to 1:

In [ ]:
M.dydx("female").summary()

Being female is associated with a 1.8 percentage point decrease in the predicted probability of voting. This is a discrete change, not a derivative.

## Subgroup marginal effects with interactions

Our model includes an interaction between `age` and `female`. We can compute the marginal effect of age separately for men and women using `atexog`:

In [ ]:
M.dydx("age", atexog={"female": [0, 1]}).summary()

The marginal effect of age is slightly larger for males (0.0128) than for females (0.0122). The difference is small but the interaction term in the model allows these effects to differ.

You can also combine multiple `atexog` variables. Here we compute the effect of education by gender:

In [ ]:
M.dydx("educ", atexog={"female": [0, 1]}).summary()

## Summary table: when to use each type

| Type | Code | Best for |
|------|------|----------|
| AME | `M.dydx("age")` | Reporting a single main effect |
| MEM | `M.dydx("age", at="mean")` | Comparing effects at a reference point |
| MER | `M.dydx("age", atexog={"age": [...]})` | Showing how effects vary |

## Recap

In this tutorial we covered:

1. **AME**: average marginal effect across the sample
2. **MEM**: marginal effect at the mean covariate profile
3. **MER**: marginal effect at user-specified values
4. **Discrete contrasts** for categorical variables like `educ`
5. **Discrete change** for binary variables like `female`
6. **Subgroup effects** using `atexog` to condition on interaction partners

## Next steps

- Learn how to obtain different kinds of standard errors in {doc}`Tutorial 4: Inference and Standard Errors </tutorials/inference>`
- Read the reference for the {doc}`dydx() method </api>`
- Learn about elasticities in {doc}`How-To: Elasticities </howto/elasticities>`
- Learn about subgroup-specific marginal effects in {doc}`How-To: Subgroup-Specific Marginal Effects </howto/subgroup_analysis>`
- Learn about custom transforms and scales in {doc}`How-To: Custom Transforms and Scales </howto/custom_transforms>`
- Understand the distinction between discrete and continuous approaches in {doc}`Explanation: Discrete vs Continuous </explanations/discrete_vs_continuous>`